# treader narration with Kokoro

Turns the chapters you exported from treader (**Download text for narration**, a `*-narration.zip`) into one MP3 per chapter, named `ch04.mp3` and so on, ready for **Upload MP3s** on the same page.

**Before you run:** use a GPU.
- **Colab:** Runtime → Change runtime type → T4 GPU.
- **Kaggle:** Settings → Accelerator → GPU T4, and Internet on.
  Add the zip with *Add Input → Upload*. Kaggle unzips it for you.

Then run every cell from the top, in order.

In [ ]:
# 1. Install Kokoro and the tools it needs (about a minute)
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null
!pip -q install "kokoro>=0.9.4" soundfile

In [ ]:
# 2. Settings
VOICE = "af_heart"   # American: af_heart, af_bella, am_michael, am_fenrir · British: bf_emma, bm_george, bm_fable
SPEED = 1.0          # 0.9 is a little slower, 1.1 a little faster
BITRATE = "48k"      # speech quality: about 6 MB for a 17-minute chapter
LANG = "b" if VOICE[0] == "b" else "a"

In [ ]:
# 3. Find the chapters: an uploaded zip (Colab), a Kaggle input, or a zip next to this notebook
import glob, os, zipfile

txts = sorted(glob.glob("/kaggle/input/**/ch*.txt", recursive=True))
if not txts:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("*-narration.zip")
    if not zips:
        try:
            from google.colab import files
            uploaded = files.upload()   # choose your *-narration.zip
            zips = [name for name in uploaded if name.endswith(".zip")]
        except ImportError:
            raise SystemExit("Add your *-narration.zip as an input (Kaggle) or next to this notebook.")
    zipfile.ZipFile(zips[0]).extractall("text")
    txts = sorted(glob.glob("text/**/ch*.txt", recursive=True))
print(len(txts), "chapters:", [os.path.basename(t) for t in txts])

In [ ]:
# 4. Narrate. Each chapter starts with its number and title; paragraphs get a short pause, scene breaks a longer one.
import re, subprocess
import numpy as np, soundfile as sf
from kokoro import KPipeline

pipe = KPipeline(lang_code=LANG)
SR = 24000
silence = lambda sec: np.zeros(int(SR * sec), dtype=np.float32)

def speak(text):
    parts = []
    for _, _, audio in pipe(text, voice=VOICE, speed=SPEED):
        if audio is not None:
            parts.append(audio.cpu().numpy() if hasattr(audio, "cpu") else np.asarray(audio, dtype=np.float32))
    return np.concatenate(parts) if parts else silence(0.1)

os.makedirs("mp3", exist_ok=True)
for path in txts:
    raw = open(path, encoding="utf-8").read()
    head = re.match(r"#\s*(\d+)\s*\|\s*(.*)", raw)
    number, title = int(head.group(1)), head.group(2).strip()
    pieces = [speak(f"Chapter {number}. {title}."), silence(1.2)]
    for para in re.split(r"\n\s*\n", raw[head.end():].strip()):
        para = para.strip()
        if not para:
            continue
        if para == "* * *":
            pieces.append(silence(1.6))
            continue
        pieces += [speak(para), silence(0.45)]
    wav = f"mp3/ch{number:02d}.wav"
    sf.write(wav, np.concatenate(pieces), SR)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-ac", "1", "-b:a", BITRATE, wav[:-4] + ".mp3"], check=True)
    os.remove(wav)
    print(f"chapter {number}: {os.path.getsize(wav[:-4] + '.mp3') / 1e6:.1f} MB")

In [ ]:
# 5. Collect the MP3s. Colab downloads the zip; on Kaggle it is in the Output panel (/kaggle/working).
import shutil
out = shutil.make_archive("treader-audio", "zip", "mp3")
print("Ready:", out, "- unzip it and upload the chNN.mp3 files on the book's page in treader.")
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass